In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Victor\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import json
import platform
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from threadpoolctl import threadpool_limits

SEED = 42
DATE_COLUMNS = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
EVENT_TYPES = ['item_view', 'search_results_view', 'photo_swipe', 'favorite_add',
               'seller_page_view', 'contact_phone_show', 'captcha_shown', 'login',
               'contact_chat_open', 'contact_message_sent']

DATA_DIR = Path('data')
OUTPUT_DIR = Path('output')


Данные: 
Удаляются только полные дубликаты событий. Совпадение одного timestamp не считается дубликатом
События берутся только внутри окна: window_start_ts <= event_ts < window_end_ts. Событие на правой границе уже не входит в окно.


In [3]:
def load_data(data_dir):
    train = pd.read_csv(data_dir / 'train.csv', parse_dates=DATE_COLUMNS)
    test = pd.read_csv(data_dir / 'test.csv', parse_dates=DATE_COLUMNS)
    events = pd.read_csv(data_dir / 'events.csv.gz', parse_dates=['event_ts'])
    for meta in (train, test):
        assert meta.cookie_id.notna().all() and meta.cookie_id.is_unique
        assert meta[DATE_COLUMNS].notna().all().all()
        assert (meta.window_end_ts > meta.window_start_ts).all()
        assert (meta.cookie_created_at <= meta.window_end_ts).all()
    assert set(train.target.unique()) == {0, 1}
    assert not set(train.cookie_id) & set(test.cookie_id)
    assert events[['cookie_id', 'event_ts']].notna().all().all()
    return train, test, events


def window_events(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']],
                      on='cookie_id', how='inner', validate='many_to_one')
    return ev.loc[ev.event_ts.ge(ev.window_start_ts) &
                  ev.event_ts.lt(ev.window_end_ts)].copy()

Признаки: 
Используются количества и доли событий, разнообразие объявлений и запросов, статистики интервалов, активные минуты, концентрация просмотров, платформы, пропуски и возраст куки. Отдельно проверяются пять признаков семейств User-Agent. Полные строки UA, тексты запросов и идентификаторы не передаются модели. Нулём заполняются отсутствующие счетчики,неопределённые статистики остаются NaN. Каждая кука обрабатывается независимо, без target и статистик по другим кукам

Интервалы и активные минуты описывают ритм действий, разнообразие объявлений - широту просмотра, доли событий - структуру поведения. Предполагается, что эти характеристики могут отличаться у ботов и людей.


In [4]:
def make_features(events, meta, rich=True):
    ev = window_events(events, meta)
    g = ev.groupby('cookie_id', sort=False)
    f = pd.DataFrame({'n_events': g.size(), 'item_nunique': g.item_id.nunique()})
    if not rich:
        return meta[['cookie_id']].merge(f, on='cookie_id', how='left',
                                         validate='one_to_one').fillna(0)

    # Сортируем события перед расчетом интервалов внутри каждой куки
    ev = ev.sort_values(['cookie_id', 'event_ts'], kind='mergesort').copy()
    ev['gap_s'] = ev.groupby('cookie_id').event_ts.diff().dt.total_seconds()
    ev['hour'] = ev.event_ts.dt.hour
    ev['minute'] = ev.event_ts.dt.floor('min')
    # Объединяем разные написания одной платформы
    ev['platform_norm'] = (ev.platform.fillna('unknown').str.lower()
                           .replace({'desktop': 'web', 'iphone': 'ios'}))
    ua = ev.user_agent.fillna('').str.lower()

    patterns = {'automation': r'headless|python|curl|wget|scrapy|bot',
                'firefox': r'firefox', 'chromium': r'chrome|chromium',
                'avito_app': r'^avito/', 'mobile': r'android|iphone|mobile'}
    for name, pattern in patterns.items():
        ev['ua_' + name] = ua.str.contains(pattern, regex=True).astype(float)
    ev['pointer_present'] = ev.pointer_x.notna() & ev.pointer_y.notna()
    g = ev.groupby('cookie_id', sort=False)
    # Для категориальных полей считаем число разных значений
    for col in ['item_category', 'item_location', 'search_query', 'platform_norm',
                'seller_type']:
        f[col + '_nunique'] = g[col].nunique()
    # Доля пропусков также используется как признак
    for col in ['item_id', 'search_query', 'pointer_x', 'pointer_y', 'seller_type']:
        f[col + '_missing_share'] = g[col].agg(lambda s: s.isna().mean())
    for col in ['search_page', 'pointer_x', 'pointer_y']:
        a = g[col].agg(['mean', 'max', 'std', 'nunique'])
        f = f.join(a.add_prefix(col + '_'))
    for col in ['ua_' + name for name in patterns] + ['pointer_present']:
        f[col + '_share'] = g[col].mean()
    for value in ['web', 'android', 'ios']:
        f['platform_' + value + '_share'] = (ev.platform_norm.eq(value)
                                             .groupby(ev.cookie_id).mean())

    # Количества и доли разных типов действий
    counts = pd.crosstab(ev.cookie_id, ev.event_name).reindex(
        columns=EVENT_TYPES, fill_value=0)
    f = f.join(counts.add_prefix('count_'))
    f = f.join(counts.div(f.n_events, axis=0).add_prefix('share_'))
    f['active_hours'] = g.hour.nunique()
    f['active_minutes'] = g.minute.nunique()
    f['max_events_per_minute'] = ev.groupby(['cookie_id', 'minute']).size().groupby(level=0).max()
    f['night_share'] = ev.hour.lt(6).groupby(ev.cookie_id).mean()
    f['activity_span_s'] = (g.event_ts.max() - g.event_ts.min()).dt.total_seconds()
    gaps = g.gap_s.agg(['mean', 'median', 'std', 'min', 'max'])
    f = f.join(gaps.add_prefix('gap_'))
    # У первого события нет предыдущего, поэтому его исключаем из расчета долей интервалов
    intervals = ev.loc[ev.gap_s.notna()]
    for limit in [1, 5, 60]:
        f[f'gap_le_{limit}_share'] = intervals.gap_s.le(limit).groupby(intervals.cookie_id).mean()
    f['gap_cv'] = f.gap_std / f.gap_mean.replace(0, np.nan)
    f['events_per_active_minute'] = f.n_events / f.active_minutes
    f['unique_item_share'] = f.item_nunique / g.item_id.count().replace(0, np.nan)
    f['top_item_share'] = (ev.groupby(['cookie_id', 'item_id']).size()
                           .groupby(level=0).max() / g.item_id.count().replace(0, np.nan))
    f['first_event_delay_s'] = (g.event_ts.min() - meta.set_index('cookie_id').window_start_ts).dt.total_seconds()
    f['last_event_to_end_s'] = (meta.set_index('cookie_id').window_end_ts - g.event_ts.max()).dt.total_seconds()

    result = meta[['cookie_id']].merge(f, on='cookie_id', how='left', validate='one_to_one')
    result['cookie_age_days'] = (meta.window_end_ts - meta.cookie_created_at).dt.total_seconds().to_numpy() / 86400
    # Отсутствующие счетчики заполняем нулями, неопределенные статистики оставляем NaN
    zeros = ['n_events', 'item_nunique', 'active_hours', 'active_minutes'] + ['count_' + x for x in EVENT_TYPES]
    result[zeros] = result[zeros].fillna(0)
    result = result.replace([np.inf, -np.inf], np.nan)
    return result

Метрика и модели:
Одинаковые score обрабатываются одной группой. Дополнительно считаются average precision и ROC-AUC. Baseline - Random Forest на двух признаках из quickstart. Улучшенные варианты - HistGradientBoostingClassifier без UA и с UA, с одинаковыми фиксированными параметрами.

In [5]:
TARGET_RECALL = 0.70


def pr_curve(y_true, score) -> tuple[np.ndarray, np.ndarray]:
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    if y_true.shape != score.shape:
        raise ValueError("y_true и score разной длины")

    n_pos = int(y_true.sum())
    if n_pos == 0:
        return np.array([]), np.array([])

    order = np.argsort(-score, kind="mergesort")
    y, s = y_true[order], score[order]

    tp = np.cumsum(y)
    k = np.arange(1, len(y) + 1)
    ends = np.r_[s[1:] != s[:-1], True]
    return tp[ends] / k[ends], tp[ends] / n_pos


def precision_at_recall(y_true, score, recall: float = TARGET_RECALL) -> float:
    prec, rec = pr_curve(y_true, score)
    if len(prec) == 0:
        return float("nan")
    ok = rec >= recall
    return float(prec[ok].max()) if ok.any() else 0.0

In [6]:
def make_model(name):
    if name == 'baseline_rf':
        return RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                      random_state=0, n_jobs=2)
    return HistGradientBoostingClassifier(max_iter=250, learning_rate=0.06,
                                          max_leaf_nodes=15, min_samples_leaf=30,
                                          l2_regularization=3.0,
                                          early_stopping=False, random_state=SEED)


def evaluate(y, p):
    return {'precision_at_recall_070': precision_at_recall(y, p),
            'average_precision': float(average_precision_score(y, p)),
            'roc_auc': float(roc_auc_score(y, p))}

Временная валидация:

Train: 6-19 апреля 2026, test: 20-26 апреля. Основное обучение - до 17 апреля, валидация - 17-19 апреля, как в quickstart. Выбор варианта - по P@R≥0.7, при равенстве - по average precision. Baseline сохраняет исходные дубликаты quickstart. Поэтому сравнение с расширенным вариантом оценивает изменение всего процесса, а не отдельно вклад очистки, признаков или модели. Два бустинга отличаются только признаками UA.

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train, test, events = load_data(DATA_DIR)
print('Размеры:', train.shape, test.shape, events.shape, flush=True)
audit = {'train_cookies': len(train), 'test_cookies': len(test),
         'raw_events': len(events), 'exact_duplicate_events': int(events.duplicated().sum())}

# Удаляем только полные дубликаты
clean = events.drop_duplicates().copy()
meta = pd.concat([train.drop(columns='target'), test], ignore_index=True)
audit['events_outside_windows_before_dedup'] = len(events) - len(window_events(events, meta))
audit['clean_events_in_windows'] = len(window_events(clean, meta))
print('Аудит:', audit, flush=True)

# Baseline сохраняет поведение quickstart, включая исходные дубликаты.
basic = make_features(events, train, rich=False).drop(columns='cookie_id')
rich = make_features(clean, train).drop(columns='cookie_id')
y = train.target.to_numpy()
valid = train.window_start_ts.ge('2026-04-17').to_numpy()
print('Обучение/валидация:', int((~valid).sum()), int(valid.sum()),
      'ботов в валидации:', int(y[valid].sum()), flush=True)

behavior_cols = [c for c in rich if not c.startswith('ua_')]
experiments = {'baseline_rf': basic,
               'behavior_hgb': rich[behavior_cols], 'behavior_ua_hgb': rich}
results, predictions = [], {}

Размеры: (11091, 5) (4909, 4) (328905, 14)
Аудит: {'train_cookies': 11091, 'test_cookies': 4909, 'raw_events': 328905, 'exact_duplicate_events': 4863, 'events_outside_windows_before_dedup': 40779, 'clean_events_in_windows': 283833}
Обучение/валидация: 9140 1951 ботов в валидации: 160


In [8]:
with threadpool_limits(limits=2):
    for name, X in experiments.items():
        model = make_model(name)
        model.fit(X.loc[~valid], y[~valid])
        p = model.predict_proba(X.loc[valid])[:, 1]
        row = {'experiment': name, 'n_features': X.shape[1], **evaluate(y[valid], p)}
        results.append(row)
        predictions[name] = p
        print(row, flush=True)
    scores = pd.DataFrame(results)
    best = scores.sort_values(['precision_at_recall_070', 'average_precision'],
                               ascending=False, kind='mergesort').iloc[0].experiment
    # Дополнительная проверка выбранного подхода на более раннем периоде.
    earlier_train = train.window_start_ts.lt('2026-04-14').to_numpy()
    earlier_valid = (train.window_start_ts.ge('2026-04-14') &
                     train.window_start_ts.lt('2026-04-17')).to_numpy()
    X = experiments[best]
    check_model = make_model(best)
    check_model.fit(X.loc[earlier_train], y[earlier_train])
    earlier_score = evaluate(y[earlier_valid], check_model.predict_proba(X.loc[earlier_valid])[:, 1])
    print('Более ранний период:', earlier_score, flush=True)

{'experiment': 'baseline_rf', 'n_features': 2, 'precision_at_recall_070': 0.10247026532479414, 'average_precision': 0.1746089870059772, 'roc_auc': 0.6073963567839197}
{'experiment': 'behavior_hgb', 'n_features': 68, 'precision_at_recall_070': 0.8201438848920863, 'average_precision': 0.7858988382389571, 'roc_auc': 0.9350083752093802}
{'experiment': 'behavior_ua_hgb', 'n_features': 73, 'precision_at_recall_070': 0.8085106382978723, 'average_precision': 0.7872691496139987, 'roc_auc': 0.9363170016750418}
Более ранний период: {'precision_at_recall_070': 0.7164948453608248, 'average_precision': 0.7654481506914658, 'roc_auc': 0.9383765405864853}


Финальная модель и файл предсказаний: 
После выбора вариант обучается на всём train. Проверяются состав и порядок cookie_id, отсутствие дубликатов и пропусков, конечность score и диапазон [0, 1]. Порог в файл не передаётся.


In [9]:
with threadpool_limits(limits=2):
    # После выбора варианта обучаем его на всём train.
    if best == 'baseline_rf':
        test_X = make_features(events, test, rich=False).drop(columns='cookie_id')
    else:
        test_X = make_features(clean, test).drop(columns='cookie_id')
    test_X = test_X.reindex(columns=X.columns)
    final_model = make_model(best)
    final_model.fit(X, y)
    pred = final_model.predict_proba(test_X)[:, 1]

In [10]:
sub = pd.DataFrame({'cookie_id': test.cookie_id, 'score': pred})
assert sub.columns.tolist() == ['cookie_id', 'score']
assert len(sub) == len(test) and sub.cookie_id.is_unique
assert sub.cookie_id.tolist() == test.cookie_id.tolist()
assert np.isfinite(sub.score).all() and sub.score.between(0, 1).all()
sub.to_csv(OUTPUT_DIR / 'submission.csv', index=False, float_format='%.17g')
scores.to_csv(OUTPUT_DIR / 'experiments.csv', index=False)
val_out = pd.DataFrame({'cookie_id': train.loc[valid, 'cookie_id'], 'target': y[valid],
                        **predictions})
val_out.to_csv(OUTPUT_DIR / 'validation_predictions.csv', index=False)
report = {'python': platform.python_version(), 'seed': SEED, 'audit': audit,
          'validation_from': '2026-04-17', 'validation_cookies': int(valid.sum()),
          'validation_positives': int(y[valid].sum()), 'selected': best,
          'earlier_validation': earlier_score, 'features': list(X.columns),
          'experiments': results}
(OUTPUT_DIR / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))
print('Выбран:', best, '; сохранён', OUTPUT_DIR / 'submission.csv', flush=True)

Выбран: behavior_hgb ; сохранён output\submission.csv


### Результаты и ограничения

На основной валидации: baseline - 0.102470; бустинг без UA - 0.820144; с UA - 0.808511. Выбран вариант без UA. В лучшей допустимой точке он находит 114 из 160 положительных куки и ошибочно отмечает 25 отрицательных: recall=0.7125, precision≈0.820144.

Дополнительная проверка выбранного варианта: обучение 6–13 апреля, валидация 14–16 апреля, P@R≥0.7=0.716495. Параметры после неё не менялись.

На основной валидации всего 160 положительных примеров. Выбор из трёх вариантов на одном периоде может давать оптимистичную оценку. Принадлежность к конкретному сервису неизвестна, поэтому обобщение на новые сервисы не проверялось. Капча зависит от политики существующего антибота; координаты в том числе от устройства. Часы используются в представлении исходных данных, без предположения о локальном времени пользователя.

**Воспроизводимость:** seeds 0 (baseline) и 42 (бустинг), два потока, фиксированные зависимости.

**Источники:** выданные quickstart.ipynb и metric.py; открытые библиотеки pandas, NumPy, scikit-learn, SciPy, joblib и threadpoolctl.
